Question 1: Study Rooms

In [ ]:
import numpy as np

def create_rooms(room_lines):
    rooms = {}
    
    for i in room_lines:
        arr = i.split("|")
    
        try:
            num = int(arr[2])
        except:
            print(f"Skipped room line: {i}")
            continue

        temp = {
            "name": arr[1],
            "total": num,
            "free": num,
            "waiting": []
        }
    
        if num < 0 or (arr[0] in rooms and arr[1] != rooms[arr[0]]["name"]):
            print(f"Skipped room line: {i}")
        elif arr[0] in rooms and arr[1] == rooms[arr[0]]["name"]:
            rooms[arr[0]]["total"] += num
            rooms[arr[0]]["free"] += num
        else:
            rooms[arr[0]] = temp
    
    return (rooms)

def check_in(rooms, members, name, id, time):
    if rooms[id]["free"] > 0:
        rooms[id]["free"] -= 1
        members[name]["room"] = id
        members[name]["time"] = time
        print(f"{name} booked {id} at {time}")
    else:
        rooms[id]["waiting"].append(name)
        print(f"{name} waits for {id} (positoion {len(rooms[id]["waiting"])})")

def fee(members, name, time):
    return max(0, int(time)-int(members[name]["time"])-2)*3

def close(closing, members, rooms):
    for name in members:
        if members[name]["room"] is not None:
            pay = fee(members, name, closing)
            members[name]["paid"] += pay
            rooms[members[name]["room"]]["free"] += 1
            members[name]["room"] = None
            members[name]["time"] = None
            print(f"{name} chacked out at {closing}, fee {pay} and paid total is {members[name]["paid"]}")

def process_events(rooms, events):
    members = {}
    
    for i in events:
        temp = tuple(i.split())
        
        if len(temp) != 4:
            print(f"Malformed event {i}")
            continue
        
        time, command, name, id = temp
        
        if name not in members:
            members[name] = {
                "paid": 0,
                "room": None,
                "time": None
            }
        
        if command == "BOOK":
            if id not in rooms:
                print(f"Unknown room {id}")
            elif members[name]["room"] is not None:
                print(f"{name} is already in a room {id}")
            else:
                check_in(rooms, members, name, id, time)
        elif command == "LEAVE":
            if members[name]["room"] != id:
                print(f"{name} is not in {id}")
            else:
                pay = fee(members, name, time)
                members[name]["paid"] += pay
                members[name]["room"] = None
                members[name]["time"] = None
                print(f"{name} left {id}, fee {pay}")
                
                if len(rooms[id]["waiting"]) != 0:
                    new_member = rooms[id]["waiting"].pop(0)
                    members[new_member]["room"] = id
                    members[new_member]["time"] = time
                    print(f"{new_member} booked {id} at {time}")
                else:
                    rooms[id]["free"] += 1
        elif command == "CHECK":
                if members[name]["room"] != id:
                    print(f"{name} is not in {id}")
                else:
                    print(f"{name} in room since {members[name]["time"]}, fee so far {fee(members, name, time)}")
                    
    
    close(closing, members, rooms)
    return (members)

def total_spend(members):
    names = sorted(members.keys())
    
    p = np.array([members[name]["paid"] for name in names])
    
    if len(p) > 0:
        print(f"Mean value: {np.mean(p)}\nMax value: {np.max(p)}")
    
    mask = p > np.mean(p)
    above_mean = []
    
    for i in range(len(names)):
        if mask[i] == 1:
            above_mean.append(names[i])
    
    print(f"Members who paid more than mean value {",".join(above_mean)}")
                    
            
            

room_lines = [

"R1|Quiet|2", "R2|Group|1", "R3|Silent|x", "R1|Quiet|1", "R2|Lab|3"

]

events = [

"1 BOOK ali R1", "2 BOOK vera R1", "2 BOOK sam R2", "3 BOOK nora R2",

"3 BOOK leo R1", "4 BOOK mia R1", "5 BOOK ali R2", "6 CHECK vera R1",

"7 LEAVE sam R1", "8 LEAVE sam R2", "9 BOOK zed R9", "10 CLEAN ali R1",

"11 LEAVE ali R1", "12 LEAVE vera", "13 LEAVE vera R1", "14 BOOK eli R1"

]

closing = 20

rooms = create_rooms(room_lines)
for id in rooms:
    print(f"{id} {rooms[id]["name"]} {rooms[id]["free"]}/{rooms[id]["total"]}")  
    
members = process_events(rooms, events)

total_spend(members)
    
 
    

Question 2: Car Park

In [ ]:
import numpy as np

def create_lots(lot_lines):
    lots = {}
    
    for i in lot_lines:
        temp = tuple(i.split("|"))
        id, name, capacity = temp
        
        if not capacity.isdigit() or int(capacity) < 0:
            print(f"Skipped lot line: {i}")
            continue
        
        if id in lots and name != lots[id]["name"]:
            print(f"Skipped line {i}")
        elif id in lots and name == lots[id]["name"]:
            lots[id]["capacity"] += int(capacity)
        else:
            lots[id] = {
                "name": name,
                "capacity": int(capacity),
                "used": 0,
                "revenue": 0
            }
    return lots

def fee(parked, plate, time):
    if plate.startswith("VIP"):
        return max(0, int(time) - int(parked[plate]["time"]) - 1)
    else:
        return max(0, int(time) - int(parked[plate]["time"]) - 1)*2

def enter(lots, parked, plate, lot, time, parked_once):
    if lot not in lots:
        print(f"Unknown lot: {lot}")
    elif parked[plate]["lot"] is not None:
        print(f"{plate} is already parked in {parked[plate]["lot"]}")
    elif lots[lot]["capacity"] == 0:
        print(f"{lot} is full, {plate} turned away")
    else:
        parked[plate]["time"] = time
        parked[plate]["lot"] = lot
        lots[lot]["capacity"] -= 1
        parked_once.add(plate)
        print(f"{plate} entered {lot}")
    
def exit(lots, parked, plate, lot, time):
    if parked[plate]["lot"] != lot:
        print(f"{plate} is not parked at {lot}")
    else:
        parked[plate]["lot"] = None
        lots[lot]["capacity"] += 1
        lots[lot]["revenue"] += fee(parked, plate, time)
        print(f"{plate} left {lot}")

def find(parked, plate, lot, time):
    if parked[plate]["lot"] != lot:
        print(f"{plate} is not in {lot}")
    else:
        print(f"{plate} in {lot}, since {parked[plate]["time"]}, fee so far {fee(parked, plate, time)}")

def close(parked, closing, lots):
    for plate in parked:
        if parked[plate]["lot"] is not None:
            rev = fee(parked, plate, closing)
            lots[parked[plate]["lot"]]["capacity"] += 1
            lots[parked[plate]["lot"]]["revenue"] += rev
            print(f"{plate} removed at {closing} from {parked[plate]["lot"]}, fee {rev}")

def process_events(lots, events):
    seen = set()
    parked_once = set()
    parked = {}
    
    for text in events:
        temp = text.split()
    
        if len(temp) < 4:
            print(f"Malformed evevnt {text}")
            continue
        
        time, action, plate, lot = temp
        
        seen.add(plate)
        if plate not in parked:
            parked[plate] = {
                "time": None,
                "lot": None
            }
        
        if action == "ENTER":
            enter(lots, parked, plate, lot, time, parked_once)
        elif action == "EXIT":
            exit(lots, parked, plate, lot, time)
        elif action == "FIND":
            find(parked, plate, lot, time)
        else:
            print(f"Unknown command {action}")
    
    close(parked, closing, lots)
    return (seen, parked_once, parked)
        


lot_lines = [

"A|North|2", "B|South|1", "C|East|0", "A|North|1", "B|West|4",

"D|Roof|two", "E|Deck|2"

]

events = [

"1 ENTER 10AA A", "1 ENTER VIP7 A", "2 ENTER 20BB B", "2 ENTER 30CC A",

"3 ENTER 40DD B", "3 ENTER 10AA E", "4 FIND VIP7 A", "5 EXIT 20BB A",

"5 EXIT 20BB B", "6 ENTER 50EE C", "7 PARK 60FF A", "8 ENTER 30CC E",

"9 EXIT 30CC A", "9 ENTER VIP9 Z", "10 EXIT 10AA A", "11 ENTER 70GG A",

"12 EXIT VIP7", "13 FIND 40DD B", "14 ENTER 80HH E", "15 EXIT VIP7 A",

"16 ENTER VIP7 E"

]

closing = 20

lots = create_lots(lot_lines)
print(lots)
seen, parked_once, parked = process_events(lots, events)

lots_sorted = sorted(lots.keys())

for lot in lots_sorted:
    print(f"{lot}: {lots[lot]["revenue"]}")
    
print(f"Never parked: {",".join(sorted(list(seen-parked_once)))}")

revenues = np.array([lots[lot]["revenue"] for lot in lots_sorted])

print(f"Mean: {np.mean(revenues)}\nMax: {np.max(revenues)}")

mask = revenues > np.mean(revenues)
above_mean = []

for i in range(len(lots_sorted)):
    if mask[i] == 1:
        above_mean.append(lots_sorted[i])

print(f"Lots whose revenue is greater than a mean value {",".join(above_mean)}")

Skipped line B|West|4
Skipped lot line: D|Roof|two
10AA entered A
VIP7 entered A
20BB entered B
30CC entered A
B is full, 40DD turned away
10AA entered E
VIP7 is not in A
20BB is not parked at A
20BB is not parked at B
C is full, 50EE turned away
30CC entered E
30CC is not parked at A
Unknown lot: Z
10AA is not parked at A
A is full, 70GG turned away
Malformed evevnt 12 EXIT VIP7
40DD is not in B
E is full, 80HH turned away
VIP7 is not parked at A
E is full, VIP7 turned away
A: 0
B: 0
C: 0
E: 0
Never parked: 40DD,50EE,60FF,70GG,80HH,VIP9
Mean: 0.0
Max: 0
Lots whose revenue is greater than a mean value 


Question 3: Bike Sharing

In [14]:
import numpy as np

def process_lines(station_lines):
    lines = {}
    
    for line in station_lines:
        element = tuple(line.split("|"))
        
        id, name, docks = element
        
        if (not docks.isdigit()) or int(docks) < 0:
            print(f"Skipped line: {line}")
        elif id in lines and lines[id]["name"] != name:
            print(f"Skipped line: {line}")
        elif id in lines and lines[id]["name"] == name:
            lines[id]["capacity"] += int(docks)
            lines[id]["bikes"] += int(docks)
        else:
            lines[id] = {
                "name": name,
                "capacity": int(docks),
                "bikes": int(docks)
            }
    
    return lines

def fee(riders, rider, time):
    diff_time = int(time) - int(riders[rider]["time"])
    price = (diff_time - 1)*2
    
    if price >= 10:
        return 10
    return price

def rent(stations, riders, rider, time, station):
    if station not in stations:
        print(f"Unknown station {station}")
    elif riders[rider]["bike"] is not None:
        print(f"{rider} already has a bike")
    elif stations[station]["bikes"] == 0:
        print(f"No bikes at {station} for {rider}")
    else:
        stations[station]["bikes"] -= 1
        riders[rider]["bike"] = station
        riders[rider]["time"] = time
        print(f"{rider} rented a bike at {station}, time {time}")
        
def return_bike(stations, riders, rider, time, station):
    if station not in stations:
            print(f"Unknown station {station}")
            return 0
    elif riders[rider]["bike"] is None:
        print(f"{rider} has no bike")
        return 0
    elif stations[station]["bikes"] == stations[station]["capacity"]:
        print(f"{station} has no free doks for {rider}")
        return 0
    else:
        payment = fee(riders, rider, time)
        print(f"{rider} returnes a bike at {station}, ride {int(time) - int(riders[rider]["time"])}, fee {payment}")
        
        stations[station]["bikes"] += 1
        riders[rider]["bike"] = None
        riders[rider]["time"] = None

        return (payment)
    
def status(stations, station):
    if station not in stations:
                print(f"Unknown station {station}")
    else:
        print(f"{station} has {stations[station]["bikes"]} of {stations[station]["capacity"]} bikes")

def close_stations(closing, stations, riders, riders_payment):
    for rider in riders:
        if riders[rider]["bike"] is not None:
            riders_payment[rider]["paid"] += fee(riders, rider, closing) + 15
            print(f"{rider} kept a bike, fee {fee(riders, rider, closing) + 15}")
            
            stations[riders[rider]["bike"]]["bikes"] += 1
            riders[rider]["bike"] = None
            riders[rider]["time"] = None

def process_events(stations, events):
    riders = {}
    riders_payment = {}
    
    for event in events:
        elements = tuple(event.split())
        
        if len(elements) != 4:
            print(f"Malformed event {event}")
            continue
            
        time, command, rider, station = elements
        
        if rider not in riders:
            riders[rider] = {
                "bike": None,
                "time": None
            }
            
            riders_payment[rider] = {
                "paid": 0
            }
            
        if command == "RENT":
            rent(stations, riders, rider, time, station)
        elif command == "RETURN": 
            riders_payment[rider]["paid"] += return_bike(stations, riders, rider, time, station)
        elif command == "STATUS":
              status(stations, station)
        else:
            print(f"Unknown command: {command}")
    
    return (riders, riders_payment)

station_lines = [

"S1|Central|3", "S2|Harbor|2", "S3|Park|-1", "S2|Harbor|1",

"S1|Airport|5", "S4|Mall|1"

]

events = [

"1 RENT ali S1", "1 RENT bob S4", "2 RENT cem S4", "2 RENT ali S2",

"3 STATUS dan S4", "3 RENT dan S1", "4 RENT eva S1", "4 RENT fay S1",

"5 RETURN bob S1", "5 RENT gus S9", "6 RETURN cem S4", "6 RETURN ali S4",

"8 RETURN ali S2", "9 RETURN dan S4", "10 RETURN dan S2",

"11 STATUS ali S1", "12 FIX bob S1", "13 RETURN eva S1",

"14 RETURN bob S2", "14 RENT bob S2", "15 RENT cem S2",

"15 RETURN bob S1", "16 RETURN bob"

]

closing = 20

stations = process_lines(station_lines)

for station in stations:
    print(f"{station}, {stations[station]["name"]}, {stations[station]["capacity"]}")
    
riders, riders_payment = process_events(stations, events)

close_stations(closing, stations, riders, riders_payment)

for rider in riders_payment:
    print(f"Paid: {rider} - {riders_payment[rider]["paid"]}")

sorted_list_riders = sorted(riders_payment.keys())

paid_money = np.array([riders_payment[rider]["paid"] for rider in sorted_list_riders])
print(f"Mean: {np.mean(paid_money)}\n Max: {np.max(paid_money)}")

mask = paid_money > np.mean(paid_money)

above_mean = []
for i in range(len(mask)):
    if mask[i] == 1:
        above_mean.append(sorted_list_riders[i])

print(f"Above mean: {','.join(above_mean)}")

Skipped line: S3|Park|-1
Skipped line: S1|Airport|5
S1, Central, 3
S2, Harbor, 3
S4, Mall, 1
ali rented a bike at S1, time 1
bob rented a bike at S4, time 1
No bikes at S4 for cem
ali already has a bike
S4 has 0 of 1 bikes
dan rented a bike at S1, time 3
eva rented a bike at S1, time 4
No bikes at S1 for fay
bob returnes a bike at S1, ride 4, fee 6
Unknown station S9
cem has no bike
ali returnes a bike at S4, ride 5, fee 8
ali has no bike
S4 has no free doks for dan
S2 has no free doks for dan
S1 has 1 of 3 bikes
Unknown command: FIX
eva returnes a bike at S1, ride 9, fee 10
bob has no bike
bob rented a bike at S2, time 14
cem rented a bike at S2, time 15
bob returnes a bike at S1, ride 1, fee 0
Malformed event 16 RETURN bob
cem kept a bike, fee 23
dan kept a bike, fee 25
Paid: ali - 8
Paid: bob - 6
Paid: cem - 23
Paid: dan - 25
Paid: eva - 10
Paid: fay - 0
Paid: gus - 0
Mean: 10.285714285714286
 Max: 25
Above mean: cem,dan


Question 4: Shop Stock

In [8]:
import numpy as np

def process_lines(product_lines):
    products = {}
    
    for product in product_lines:
        properties = tuple(product.split("|"))
    
        id, name, stock, price = properties
        
        if not stock.isdigit():
            print(f"Skipped line {product}")
            continue
        
        products[id] = {
            "name": name,
            "stock": int(stock),
            "price": int(price),
            "revenue": 0
        }
    for id in products:
        print(f"{id} {products[id]["name"]} stock {products[id]["stock"]} price {products[id]["price"]}")
    
    return products

def sell(products, sold, id, value):
    if products[id]["stock"] < int(value):
        print(f"Not enough {products[id]["name"]}: wanted {value}, have {products[id]["stock"]}")
        return

    products[id]["stock"] -= int(value)
    products[id]["revenue"] += int(value) * products[id]["price"]
    sold.add(id)
    print(f"Sold {value} {products[id]["name"]} for {int(value) * products[id]["price"]}")
    
def adjust(products, id, value):
    if value.startswith("+"):
        products[id]["stock"] += int(value[1:])
    else:
        products[id]["stock"] -= int(value[1:])
        
    print(f"{products[id]["name"]} stock is now {products[id]["stock"]}")

def process_events(products, events):
    sold = set()
    
    for event in events:
        properties = tuple(event.split())
        
        time, command, id, value = properties
        
        if command == "SELL":
            sell(products, sold, id, value)
        elif command == "ADJUST":
            adjust(products, id, value)
    
    return sold

product_lines = [

"P1|Pen|20|2", "P2|Book|5|30", "P3|Bag|abc|50", "P4|Ruler|0|5", "P5|Mug|4|6"

]

events = [

"1 SELL P1 8", "1 SELL P2 6", "2 SELL P2 5", "2 ADJUST P4 +10",

"3 SELL P4 3", "4 ADJUST P1 -9", "5 SELL P1 4"

]

low_limit = 4

products = process_lines(product_lines)
print()
sold = process_events(products, events)
print()

keys = sorted(products.keys())

for id in keys:
    print(f"Stock: {id}, {products[id]["name"]}, {products[id]["stock"]}")
    
low_stock = [products[id]["name"] for id in products if products[id]["stock"] < low_limit]
print(f"Low stock: {', '.join(low_stock)}")

print(f"Never sold: {set(keys) - sold}")

for id in keys:
    print(f"Revenue: {products[id]["name"]} {products[id]["revenue"]}")
    
revenues = np.array([products[id]["revenue"] for id in keys])
print(f"Mean: {round(np.mean(revenues), 2)}\nMax: {np.max(revenues)}")

mask = revenues > np.mean(revenues)
above_mean = []

for i in range(len(mask)):
    if mask[i] == 1:
        above_mean.append(products[keys[i]]["name"])
        
print(f"Above mean: {', '.join(above_mean)}")

Skipped line P3|Bag|abc|50
P1 Pen stock 20 price 2
P2 Book stock 5 price 30
P4 Ruler stock 0 price 5
P5 Mug stock 4 price 6

Sold 8 Pen for 16
Not enough Book: wanted 6, have 5
Sold 5 Book for 150
Ruler stock is now 10
Sold 3 Ruler for 15
Pen stock is now 3
Not enough Pen: wanted 4, have 3

Stock: P1, Pen, 3
Stock: P2, Book, 0
Stock: P4, Ruler, 7
Stock: P5, Mug, 4
Low stock: Pen, Book
Never sold: {'P5'}
Revenue: Pen 16
Revenue: Book 150
Revenue: Ruler 15
Revenue: Mug 0
Mean: 45.25
Max: 150
Above mean: Book


Question 5: Bank Accounts

In [10]:
import numpy as np

def process_lines(account_lines):
    accounts = {}
    
    for account in account_lines:
        properties = tuple(account.split("|"))
        id, name, balance = properties
        
        if not balance.isdigit():
            print(f"Skipped account line: {account}")
            continue
        
        accounts[id] = {
            "owner": name,
            "balance": int(balance)
        }
        
    for id in accounts:
        print(f"{id} {accounts[id]["owner"]} {accounts[id]["balance"]}")
    
    return accounts

def deposit(accounts, id, amount):
    accounts[id]["balance"] += int(amount)
    print(f"{accounts[id]["owner"]} deposited {amount}, balance {accounts[id]["balance"]}")

def withdraw(accounts, id, amount):
    if int(amount) > accounts[id]["balance"]:
        print(f"{accounts[id]["owner"]} cannot withdraw {amount}(balance {accounts[id]["balance"]})")
    else:
        accounts[id]["balance"] -= int(amount)
        print(f"{accounts[id]["owner"]} withdrew {amount}, balance {accounts[id]["balance"]}")
        
def transfer(accounts, active, target, amount):
    sender = target[0]
    receiver = target[1]
    
    if accounts[sender]["balance"] < int(amount):
        print(f"{accounts[sender]["owner"]} cannot transfer {amount}, balance {accounts[sender]["balance"]}")
    else:
        accounts[receiver]["balance"] += int(amount)
        accounts[sender]["balance"] -= int(amount)
        active.add(sender)
        active.add(receiver)
        
        print(f"{accounts[sender]["owner"]} sent {amount} to {accounts[receiver]["owner"]}, balance {accounts[sender]["balance"]}")

def process_events(accounts, events):
    active = set()
    
    for event in events:
        properties = tuple(event.split())
        time, command, target, amount = properties
        
        if command == "DEPOSIT":
            deposit(accounts, target, amount)
        elif command == "WITHDRAW":
            withdraw(accounts, target, amount)
        elif command == "TRANSFER":
            transfer(accounts, active, target.split(">"), amount)
    
    return active
            
account_lines = ["A1|Ali|100", "A2|Vera|50", "A3|Sam|-5", "A4|Leo|0", "A5|Mia|31"]

events = [

"1 DEPOSIT A1 30", "1 WITHDRAW A2 60", "2 WITHDRAW A2 20",

"2 TRANSFER A1>A4 60", "4 TRANSFER A4>A2 70", "5 TRANSFER A4>A2 25"

]

accounts = process_lines(account_lines)
print()

active = process_events(accounts, events)
print()

names = sorted([accounts[id]["owner"] for id in accounts])
ids = []
for name in names:
    for id in accounts:
        if accounts[id]["owner"] == name:
            ids.append(id)
    
for id in ids:
    print(f"Balance {accounts[id]["owner"]} {accounts[id]["balance"]}")
    
inactive = set(ids) - active
inactive_names = [accounts[id]["owner"] for id in inactive]
print(f"Inactive: {', '.join(inactive_names)}")

balances = np.array([accounts[id]["balance"] for id in ids])

print(f"Mean {round(np.mean(balances), 2)}\nMax {np.max(balances)}")

mask = balances > np.mean(balances)
above_mean = []

for i in range(len(mask)):
    if mask[i] == 1:
        above_mean.append(accounts[ids[i]]["owner"])
print(f"Above mean: {', '.join(above_mean)}")

Skipped account line: A3|Sam|-5
A1 Ali 100
A2 Vera 50
A4 Leo 0
A5 Mia 31

Ali deposited 30, balance 130
Vera cannot withdraw 60(balance 50)
Vera withdrew 20, balance 30
Ali sent 60 to Leo, balance 70
Leo cannot transfer 70, balance 60
Leo sent 25 to Vera, balance 35

Balance Ali 70
Balance Leo 35
Balance Mia 31
Balance Vera 55
Inactive: Mia
Mean 47.75
Max 70
Above mean: Ali, Vera


In [ ]:
import numpy as np

def process_lines(agent_lines):
    agents = {}
    
    for agent in agent_lines:
        properties = tuple(agent.split("|"))
        
        id, name, limit = properties
        
        if not limit.isdigit():
            print(f"Skipped agent line: {agent}")
            continue
        
        agents[id] = {
            "name": name, 
            "limit": int(limit),
            "open": 0,
            "score": 0
        }
    
    for id in agents:
        print(f"{id} {agents[id]["name"]} {agents[id]["open"]}/{agents[id]["score"]}")
    
    return agents

def open():
    for id in agents:
        

def process_events(agents, events):
    priority_of = {}
    
    for event in  events:
        properties = events.split()
        
        time, command, ticket, value = properties
        
        if command == "OPEN":
            

agent_lines = ["A1|Mia|1", "A2|Tom|2", "A3|Zoe|x", "A4|Kai|1"]

events = [

"1 OPEN T1 HIGH", "1 OPEN T2 LOW", "2 OPEN T3 LOW", "2 OPEN T4 HIGH",

"3 OPEN T5 LOW", "4 CLOSE T1 A1", "4 OPEN T6 LOW", "5 CLOSE T3 A2",

"6 CLOSE T4 A4", "7 CLOSE T6 A1"

]

agents = process_lines(agent_lines)